In [43]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [44]:
# ==========================================
# 1. Load the Data
# ==========================================
weekly = pd.read_csv('..\\data\\weekly.csv')
transactions = pd.read_csv('..\\data\\transactions.csv')
people = pd.read_csv('..\\data\\people.csv')

In [ ]:
transactions.drop("week", axis=1, inplace=True)
transactions.columns
pd.DataFrame.to_csv()


Index(['person_id', 'date', 'amount', 'category', 'merchant', 'balance_after'], dtype='str')

In [6]:
# ==========================================
# 2. Feature Engineering: Transactions
# ==========================================
# Calculate total spending and number of transactions per person per week
tx_agg = transactions.groupby(['person_id', 'week']).agg(
    total_spent=('amount', 'sum'),
    tx_count=('amount', 'count')
).reset_index()
# Pivot spending by category to create features like 'spend_groceries', 'spend_transport'
tx_cat = transactions.pivot_table(
    index=['person_id', 'week'], 
    columns='category', 
    values='amount', 
    aggfunc='sum', 
    fill_value=0
).reset_index()

# Rename categorical columns to avoid spaces/issues
tx_cat.columns = [f"spend_{col}" if col not in ['person_id', 'week'] else col for col in tx_cat.columns]


In [7]:
# ==========================================
# 3. Merge Datasets
# ==========================================
# Merge static person data into weekly tracker
df = pd.merge(weekly, people, on='person_id', how='left')

# Merge engineered transaction features
df = pd.merge(df, tx_agg, on=['person_id', 'week'], how='left')
df = pd.merge(df, tx_cat, on=['person_id', 'week'], how='left')

# Weeks with no transactions will have NaNs after merging; fill with 0
df = df.fillna(0) 

In [8]:
# ==========================================
# 4. Feature Engineering: Financial Ratios
# ==========================================
# Only train on rows where the 6-week window actually completed
df = df[df['label_complete'] == 1]

# Create standard risk indicators
df['rent_to_income_ratio'] = df['monthly_rent'] / df['monthly_income']
# Adding a tiny number (1e-5) to avoid dividing by zero if overdraft limit is 0
df['credit_utilization'] = df['balance'] / (df['overdraft_limit'] + 1e-5) 
df['available_liquidity'] = df['balance'] + df['overdraft_limit']

# Convert categorical 'persona' to numeric values for XGBoost
le = LabelEncoder()
if 'persona' in df.columns:
    df['persona_encoded'] = le.fit_transform(df['persona'].astype(str))


In [9]:
# ==========================================
# 5. Model Setup & Train-Test Split
# ==========================================
target = 'crisis_within_6w'

# Drop columns that are identifiers, dates, the target, or leak future info
cols_to_drop = [
    'person_id', 'week', 'week_start', target, 'label_complete', 'persona', 'shock_week'
]
features = [col for col in df.columns if col not in cols_to_drop]

X = df[features]
y = df[target]

# Stratify ensures the same proportion of bankruptcies in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [14]:
X_test.columns

Index(['balance', 'missed_rent', 'already_missed_rent', 'monthly_income',
       'monthly_rent', 'overdraft_limit', 'total_spent', 'tx_count',
       'spend_bnpl_repayment', 'spend_council_tax', 'spend_discretionary',
       'spend_energy', 'spend_failed_payment', 'spend_gambling',
       'spend_groceries', 'spend_overdraft_fee', 'spend_payday_loan',
       'spend_payday_loan_repayment', 'spend_phone', 'spend_rent',
       'spend_salary', 'spend_transport', 'spend_universal_credit',
       'rent_to_income_ratio', 'credit_utilization', 'available_liquidity',
       'persona_encoded'],
      dtype='str')

In [10]:
# ==========================================
# 6. Train the XGBoost Model
# ==========================================
# Calculate weights to handle rare events (imbalanced data)
# Ratio of negative class to positive class
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train) if sum(y_train) > 0 else 1

model = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_weight, # Forces model to pay more attention to the rare '1's
    random_state=42,
    eval_metric='auc'
)

print("Training model...")
model.fit(X_train, y_train)


Training model...


,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGB

In [ ]:
# ==========================================
# 7. Evaluate Performance
# ==========================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# ROC-AUC is often a better metric than accuracy for imbalanced datasets
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")


--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.93      0.96     16728
           1       0.58      0.96      0.73      1672

    accuracy                           0.93     18400
   macro avg       0.79      0.95      0.84     18400
weighted avg       0.96      0.93      0.94     18400

[5.0150475e-04 1.6269154e-03 3.5382359e-04 ... 4.4792065e-01 9.4816089e-04
 3.3891803e-04]
ROC-AUC Score: 0.9893


In [ ]:
weekly_csv_data = """person_id,week,week_start,balance,missed_rent,crisis_within_6w,already_missed_rent,label_complete
1,0,2026-01-05,1176.61,0,0,0,1
1,1,2026-01-12,1049.57,0,0,0,1
1,2,2026-01-19,820.10,0,0,0,1
1,3,2026-01-26,410.20,0,1,0,1
1,4,2026-02-02,-120.50,1,1,1,1
"""

people_csv_data = """person_id,persona,shock_week,monthly_income,monthly_rent,overdraft_limit
1,rent_shock,18,1833.55,832.96,1000.0
"""

transactions_csv_data = """person_id,date,week,amount,category,merchant,balance_after
1,2026-01-06,0,-44.36,groceries,Asda,1242.95
1,2026-01-06,0,-26.06,transport,Bus Fare,1216.89
1,2026-01-27,3,-120.00,bills,Energy,300.20
"""

In [ ]:
import io
weekly = pd.read_csv(io.StringIO(weekly_csv_data))
people = pd.read_csv(io.StringIO(people_csv_data))
transactions = pd.read_csv(io.StringIO(transactions_csv_data))

# ==========================================
# 2. Aggregations & Merging
# ==========================================
tx_agg = transactions.groupby(['person_id', 'week']).agg(
    total_spent=('amount', 'sum'),
    tx_count=('amount', 'count')
).reset_index()

tx_cat = transactions.pivot_table(
    index=['person_id', 'week'],
    columns='category',
    values='amount',
    aggfunc='sum',
    fill_value=0
).reset_index()

tx_cat.columns = [f"spend_{c}" if c not in ['person_id', 'week'] else c for c in tx_cat.columns]

df = pd.merge(weekly, people, on='person_id', how='left')
df = pd.merge(df, tx_agg, on=['person_id', 'week'], how='left')
df = pd.merge(df, tx_cat, on=['person_id', 'week'], how='left')
df = df.fillna(0)

# Filter complete labels
df = df[df['label_complete'] == 1].copy()

# ==========================================
# 3. Feature Engineering
# ==========================================
df['rent_to_income_ratio'] = df['monthly_rent'] / df['monthly_income']
df['credit_utilization'] = df['balance'] / (df['overdraft_limit'] + 1e-5)
df['available_liquidity'] = df['balance'] + df['overdraft_limit']

le = LabelEncoder()
df['persona_encoded'] = le.fit_transform(df['persona'].astype(str))

# ==========================================
# 4. Model Training & Probability Output
# ==========================================
target = 'crisis_within_6w'
cols_to_drop = ['person_id', 'week', 'week_start', target, 'label_complete', 'persona', 'shock_week']
features = [c for c in df.columns if c not in cols_to_drop]

X = df[features]
y = df[target]

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(y_pred, y_prob)

[0 0] [0.5 0.5]


In [24]:
people["persona"].unique()

<StringArray>
['rent_shock', 'job_loss', 'steady', 'near_miss', 'gambling']
Length: 5, dtype: str

In [20]:
print(X_test)

   balance  missed_rent  already_missed_rent  monthly_income  monthly_rent  \
2    820.1            0                    0         1833.55        832.96   
3    410.2            0                    0         1833.55        832.96   

   overdraft_limit  total_spent  tx_count  spend_bills  spend_groceries  \
2           1000.0          0.0       0.0          0.0              0.0   
3           1000.0       -120.0       1.0       -120.0              0.0   

   spend_transport  rent_to_income_ratio  credit_utilization  \
2              0.0              0.454288              0.8201   
3              0.0              0.454288              0.4102   

   available_liquidity  persona_encoded  
2               1820.1                0  
3               1410.2                0  
